In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
# Import packages and modules
import numpy as np
from RL4CRN_Feedback.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN_Feedback.Policies.BimolecularMassActionPolicy import BimolecularMassActionPolicy
from RL4CRN_Feedback.Utils.Utils import batch_multi_hot

In [3]:
# Construct the basic CRN
species_labels = ['X_1', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2', 'u_3', 'u_4']
stoichiometry_reactants = np.array([[0], [1], [0]], dtype=np.int8)
stoichiometry_products = np.array([[1], [1], [0]], dtype=np.int8)
k = 1
parameters = np.array([k], dtype=np.float32)
input_influence_matrix = np.array([[0], [0], [0], [0]], dtype=np.int8)
outputs = np.array([1], dtype=np.int8)
IOCRN_0 = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)

# Add reactions to CRN_1
IOCRN_1 = IOCRN_0.clone()
IOCRN_1.add_reaction({'reactant1 index': 0, 'reactant2 index': 1, 'product1 index': 0, 'product2 index': 0, 'input influence index': 0, 'rate constant':0.1}, mode='species index')
IOCRN_1.add_reaction({'reactant1 index': 1, 'reactant2 index': 1, 'product1 index': 2, 'product2 index': 3, 'input influence index': 3, 'rate constant':0.5}, mode='species index')
IOCRN_1.add_reaction({'reactant1 index': 1, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 3, 'input influence index': 2, 'rate constant':0.7}, mode='species index')


# Add reactions to CRN_2
IOCRN_2 = IOCRN_0.clone()
IOCRN_2.add_reaction({'reactant1 index': 2, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 1, 'input influence index': 2, 'rate constant':0.1}, mode='species index')
IOCRN_2.add_reaction({'reactant1 index': 2, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 3, 'input influence index': 4, 'rate constant':0.1}, mode='species index')
IOCRN_2.add_reaction({'reactant1 index': 0, 'reactant2 index': 1, 'product1 index': 2, 'product2 index': 2, 'input influence index': 0, 'rate constant':0.3}, mode='species index')

# Add reactions to CRN_3
IOCRN_3 = IOCRN_0.clone()
IOCRN_3.add_reaction({'reactant1 index': 0, 'reactant2 index': 0, 'product1 index': 0, 'product2 index': 3, 'input influence index': 0, 'rate constant':0.8}, mode='species index')
IOCRN_3.add_reaction({'reactant1 index': 3, 'reactant2 index': 3, 'product1 index': 1, 'product2 index': 3, 'input influence index': 3, 'rate constant':0.2}, mode='species index')
IOCRN_3.add_reaction({'reactant1 index': 0, 'reactant2 index': 2, 'product1 index': 1, 'product2 index': 1, 'input influence index': 3, 'rate constant':0.9}, mode='species index')

# Create list of CRNs to represent a batch
IOCRN_list = [IOCRN_1, IOCRN_2, IOCRN_3]

# Print details for each CRN
for i, IOCRN in enumerate(IOCRN_list):
    print(f"IOCRN_{i}:")
    print(f"Observation: (Reactions Indices, Rates, Inputs Influences) = ({IOCRN.reactions_indices}, {IOCRN.parameters}, {IOCRN.list_influenced_reactions})")
    IOCRN.print_reactions()
    print('-----------------------------------')

IOCRN_0:
Observation: (Reactions Indices, Rates, Inputs Influences) = ([23 10 44 58], [1.  0.1 0.5 0.7], [array([], dtype=uint64), array([58], dtype=uint64), array([44], dtype=uint64), array([], dtype=uint64)])
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
Reaction 1: X_1 -> 0 ; Rate Constant: 0.1 
Reaction 2: 2 X_1 -> Z_1 + Z_2 ; Rate Constant: 0.5u_3 
Reaction 3: X_1 + Z_2 -> Z_2 ; Rate Constant: 0.7u_2 

-----------------------------------
IOCRN_1:
Observation: (Reactions Indices, Rates, Inputs Influences) = ([23 74 76 16], [1.  0.1 0.1 0.3], [array([], dtype=uint64), array([74], dtype=uint64), array([], dtype=uint64), array([76], dtype=uint64)])
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
Reaction 1: Z_1 + Z_2 -> X_1 ; Rate Constant: 0.1u_2 
Reaction 2: Z_1 + Z_2 -> Z_2 ; Rate Consta

In [4]:
# Collect observations from the batch of CRNs
num_species = 3; num_inputs = 4
num_possible_reactions = IOCRN_0.get_reactions_range()
batch_size = len(IOCRN_list)
reactions_indices_batch = np.array([CRN.reactions_indices for CRN in IOCRN_list])
parameters_batch = np.array([CRN.parameters for CRN in IOCRN_list])

def get_influenced_reactions_batch(IOCRN_list, num_inputs):
    influenced_reactions = []
    for i in range(num_inputs):
        rows = [np.array(crn.list_influenced_reactions[i]) for crn in IOCRN_list]
        max_len = max((len(r) for r in rows), default=0)
        padded = [np.pad(r, (0, max_len - len(r)), constant_values=0) for r in rows]
        influenced_reactions.append(np.array(padded).astype(np.uint64))
    return influenced_reactions

reactions_indices_influenced_by_inputs_batch = get_influenced_reactions_batch(IOCRN_list, num_inputs)

# Print each observation separately
for i in range(batch_size):
    print(f"IOCRN_{i}:")
    IOCRN_list[i].print_reactions()
    print(f"Reactions Indices: {reactions_indices_batch[i]}")
    print(f"Parameters: {parameters_batch[i]}")
    for j in range(num_inputs):
        print(f"Reactions Indices Influenced by Input {j+1}:\n {reactions_indices_influenced_by_inputs_batch[j][i]}")
    print('-----------------------------------')

# Print the batch of observations
print("Batch of Observations:")
print(f"Reactions Indices:\n {reactions_indices_batch}")
print(f"Parameters:\n {parameters_batch}")
for i in range(num_inputs):
    print(f"Reactions Indices Influenced by Input {i+1}:\n {reactions_indices_influenced_by_inputs_batch[i]}")
print('-----------------------------------')

IOCRN_0:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
Reaction 1: X_1 -> 0 ; Rate Constant: 0.1 
Reaction 2: 2 X_1 -> Z_1 + Z_2 ; Rate Constant: 0.5u_3 
Reaction 3: X_1 + Z_2 -> Z_2 ; Rate Constant: 0.7u_2 

Reactions Indices: [23 10 44 58]
Parameters: [1.  0.1 0.5 0.7]
Reactions Indices Influenced by Input 1:
 []
Reactions Indices Influenced by Input 2:
 [58]
Reactions Indices Influenced by Input 3:
 [44  0]
Reactions Indices Influenced by Input 4:
 [0]
-----------------------------------
IOCRN_1:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
Reaction 1: Z_1 + Z_2 -> X_1 ; Rate Constant: 0.1u_2 
Reaction 2: Z_1 + Z_2 -> Z_2 ; Rate Constant: 0.1u_4 
Reaction 3: X_1 -> 2 Z_1 ; Rate Constant: 0.3 

Reactions Indices: [23 74 76 16]
Parameters: [1.  0.1 0.1 0.3]
Reactions Indices Influenced b

In [ ]:
# Compute the multi-hot encoding of the observations
reactions_indices_batch_hot, rates_batch_hot = batch_multi_hot(reactions_indices_batch, num_possible_reactions, parameters_batch)
reactions_indices_influenced_by_inputs_batch_hot = [batch_multi_hot(reactions_indices_influenced_by_inputs_batch[i], num_possible_reactions) for i in range(num_inputs)]
print('Shape of reactions_indices_batch_hot:', reactions_indices_batch_hot.shape)
print('Shape of parameters_batch:', parameters_batch.shape)
for i in range(batch_size):
    print(f"Reactions Indices for CRN_{i}: {reactions_indices_batch[i]} \nHot Encoding: \n{reactions_indices_batch_hot[i]}")
    print(f"Rates for CRN_{i}: {parameters_batch[i]} \nHot Encoding: \n{rates_batch_hot[i]}")
    for j in range(num_inputs):
        print(f"Reactions Indices Influenced by Input {j+1} for CRN_{i}: {reactions_indices_influenced_by_inputs_batch[j][i]} \nHot Encoding: \n{reactions_indices_influenced_by_inputs_batch_hot[j][i]}")
    print('-----------------------------------')


Shape of reactions_indices_batch_hot: torch.Size([3, 91])
Shape of parameters_batch: (3, 4)
Reactions Indices for CRN_0: [23 10 44 58] 
Hot Encoding: 
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0.])
Rates for CRN_0: [1.  0.1 0.5 0.7] 
Hot Encoding: 
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.1000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0